In [36]:
def display_board(board):
    """Prints the board in the required +-------+ format."""
    print("+-------+-------+-------+")
    for row in board:
        print("|       |       |       |")
        print(f"|   {row[0]}   |   {row[1]}   |   {row[2]}   |")
        print("|       |       |       |")
        print("+-------+-------+-------+")

In [37]:
def make_list_of_free_fields(board):
    """Returns a list of tuples (row, col) for all empty squares."""
    free_fields = []
    for r in range(3):
        for c in range(3):
            # If the square contains a digit (1-9), it is free
            if board[r][c] not in ['X', 'O']:
                free_fields.append((r, c))
    return free_fields

In [38]:
def victory_for(board, sign):
    """Checks if the player with 'sign' has 3 in a row."""
    # Check Rows
    for r in range(3):
        if board[r][0] == board[r][1] == board[r][2] == sign:
            return True
    # Check Columns
    for c in range(3):
        if board[0][c] == board[1][c] == board[2][c] == sign:
            return True
    # Check Diagonals
    if board[0][0] == board[1][1] == board[2][2] == sign:
        return True
    if board[0][2] == board[1][1] == board[2][0] == sign:
        return True
    
    return False

In [39]:
from random import randrange
def enter_move(board):
    """Asks the user for their move, validates it, and updates the board."""
    while True:
        try:
            move = int(input("Enter your move (1-9): "))
            if move < 1 or move > 9:
                print("Error: Field number must be between 1 and 9.")
                continue
            
            # Convert 1-9 to row/col indices
            row = (move - 1) // 3
            col = (move - 1) % 3
            
            # Check if the square is occupied by 'X' or 'O'
            if board[row][col] in ['X', 'O']:
                print("Error: That field is already occupied! Try again.")
                continue
                
            board[row][col] = 'O' # Human is always 'O'
            break 
        except ValueError:
            print("Error: Please enter a valid integer.")

In [40]:
def draw_move_random(board):
    """The computer makes a random move using randrange."""
    free_fields = make_list_of_free_fields(board)
    
    if free_fields:
        # Pick a random index from the list of available (row, col) tuples
        random_index = randrange(len(free_fields))
        row, col = free_fields[random_index]
        
        board[row][col] = 'X' # Computer is always 'X'
        print(f"Computer (Random) chose square: {(row * 3) + col + 1}")

In [41]:
def minimax(board, depth, is_maximizing):
    """The recursive engine that calculates the best possible move."""
    # Base cases: Does someone win in this imaginary scenario?
    if victory_for(board, 'X'):
        return 1
    if victory_for(board, 'O'):
        return -1
    if not make_list_of_free_fields(board):
        return 0

    if is_maximizing:
        best_score = -float('inf')
        for row, col in make_list_of_free_fields(board):
            temp_val = board[row][col] # Save current value (the number)
            board[row][col] = 'X'      # Try the move
            score = minimax(board, depth + 1, False)
            board[row][col] = temp_val # Undo the move (Backtrack)
            best_score = max(score, best_score)
        return best_score
    else:
        best_score = float('inf')
        for row, col in make_list_of_free_fields(board):
            temp_val = board[row][col]
            board[row][col] = 'O'
            score = minimax(board, depth + 1, True)
            board[row][col] = temp_val
            best_score = min(score, best_score)
        return best_score

In [42]:
def draw_move_intelligent(board):
    """The computer uses Minimax to find the optimal move."""
    best_score = -float('inf')
    best_move = None
    
    print("Computer (AI) is calculating the best move...")
    
    for row, col in make_list_of_free_fields(board):
        temp_val = board[row][col]
        board[row][col] = 'X'
        score = minimax(board, 0, False)
        board[row][col] = temp_val # Undo
        
        if score > best_score:
            best_score = score
            best_move = (row, col)
            
    if best_move:
        r, c = best_move
        board[r][c] = 'X'

In [43]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. INITIALIZE THE DATA (The Board)
board = [[str(3 * j + i + 1) for i in range(3)] for j in range(3)]
board[1][1] = 'X'  # Computer starts in the middle

# 2. CREATE THE BUTTONS (The UI)
# We create the list of buttons FIRST so the functions can see them
buttons = [widgets.Button(description=str(i+1), 
                          layout=widgets.Layout(width='60px', height='60px')) 
           for i in range(9)]

output = widgets.Output()

# 3. DEFINE THE VISUAL SYNC FUNCTION
def sync_visuals():
    """Updates all 9 buttons to match the current 'board' list."""
    for i in range(9):
        r, c = i // 3, i % 3
        buttons[i].description = board[r][c]
        if board[r][c] == 'X':
            buttons[i].style.button_color = 'salmon'
        elif board[r][c] == 'O':
            buttons[i].style.button_color = 'lightblue'
        else:
            buttons[i].style.button_color = None

# 4. DEFINE THE CLICK LOGIC
def on_button_clicked(b):
    with output:
        # Get the number from the button clicked
        idx = int(b.description) - 1 if b.description.isdigit() else -1
        if idx == -1: return # Square already taken
        
        row, col = idx // 3, idx % 3
        
        # Human Move
        board[row][col] = 'O'
        sync_visuals()
        
        if victory_for(board, 'O'):
            print("🎉 You won!")
            for btn in buttons: btn.disabled = True
            return
        
        if not make_list_of_free_fields(board):
            print("🤝 Tie!")
            return

        # Computer Move (Using your Intelligent logic from the previous cells)
        draw_move_intelligent(board) 
        sync_visuals()
        
        if victory_for(board, 'X'):
            print("🤖 Computer wins!")
            for btn in buttons: btn.disabled = True

# 5. LINK EVERYTHING AND DISPLAY
for b in buttons:
    b.on_click(on_button_clicked)

sync_visuals() # This makes Square 5 turn red immediately
grid = widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat(3, 70px)"))

print("Tic-Tac-Toe AI Mode: Click a number to play as 'O'")
display(grid, output)

Tic-Tac-Toe AI Mode: Click a number to play as 'O'


GridBox(children=(Button(description='1', layout=Layout(height='60px', width='60px'), style=ButtonStyle()), Bu…

Output()

In [44]:
# def main():
#     """Main menu to choose the game mode."""
#     # 1. Initialize the 3x3 board
#     board = [[str(3 * j + i + 1) for i in range(3)] for j in range(3)]
    
#     print("========================================")
#     print("   WELCOME TO AI LAB: TIC-TAC-TOE")
#     print("========================================")
#     print("1. Human vs. Random Computer")
#     print("2. Human vs. Intelligent Agent (Minimax)")
#     print("3. Human vs. Human")
    
#     choice = input("\nSelect Mode (1, 2, or 3): ")

#     # Requirement: Computer always starts in the middle for AI modes
#     if choice in ['1', '2']:
#         board[1][1] = 'X'
#         print("\nComputer starts in the middle (Square 5).")
    
#     display_board(board)

#     while True:
#         # --- PLAYER O TURN (Always Human) ---
#         print("\n[Player O's Turn]")
#         enter_move(board)
#         display_board(board)
#         if victory_for(board, 'O'):
#             print("🎉 Player O Wins!")
#             break
#         if not make_list_of_free_fields(board):
#             print("🤝 It's a Tie!")
#             break

#         # --- PLAYER X TURN ---
#         print("\n[Player X's Turn]")
#         if choice == '1':
#             draw_move_random(board)
#         elif choice == '2':
#             draw_move_intelligent(board)
#         elif choice == '3':
#             # For Human vs Human, we need a small tweak to enter_move
#             # or just call a version of it for 'X'
#             print("Player X, please enter your move.")
#             # (You can reuse enter_move logic here for X)
#             enter_move_pvp(board, 'X') 
        
#         display_board(board)
#         if victory_for(board, 'X'):
#             print("🎉 Player X Wins!")
#             break
#         if not make_list_of_free_fields(board):
#             print("🤝 It's a Tie!")
#             break

# # Helper for Human vs Human mode
# def enter_move_pvp(board, sign):
#     while True:
#         try:
#             move = int(input(f"Enter move for {sign} (1-9): "))
#             row, col = (move - 1) // 3, (move - 1) % 3
#             if board[row][col] not in ['X', 'O']:
#                 board[row][col] = sign
#                 break
#             else: print("Occupied!")
#         except: print("Invalid input.")

# # Run the game
# if __name__ == "__main__":
#     main()